In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json



cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d michelecarissimi4/reduced-codesearchnet


Dataset URL: https://www.kaggle.com/datasets/michelecarissimi4/reduced-codesearchnet
License(s): unknown
 98% 535M/547M [00:08<00:00, 135MB/s]
100% 547M/547M [00:08<00:00, 68.5MB/s]


In [3]:
!unzip reduced-codesearchnet.zip


Archive:  reduced-codesearchnet.zip
  inflating: go_selected_data.json   
  inflating: java_selected_data.json  
  inflating: javascript_selected_data.json  
  inflating: php_selected_data.json  
  inflating: python_selected_data.json  
  inflating: ruby_selected_data.json  


In [5]:
!ls

go_selected_data.json	       php_selected_data.json	  ruby_selected_data.json
javascript_selected_data.json  python_selected_data.json  sample_data
java_selected_data.json        reduced-codesearchnet.zip


In [6]:
import pandas as pd

df = pd.DataFrame({
    'cleaned_code': [
        'def export_probabilityoutputname, kwargs\n    ...',
        'def shared_symbol_tablename, version, symbols,...',
        'def _read_para_hip_transport_modeself, code, c...',
        'def setCheckable self, state \n        \n     ...',
        'def from_moleculecls, molecule, labelsNone\n  ...'
    ]
})

print(df['cleaned_code'].to_list())


['def export_probabilityoutputname, kwargs\n    ...', 'def shared_symbol_tablename, version, symbols,...', 'def _read_para_hip_transport_modeself, code, c...', 'def setCheckable self, state \n        \n     ...', 'def from_moleculecls, molecule, labelsNone\n  ...']


In [7]:
def function_name(param1, param2):
    print(param1, param2)

In [8]:
import re

def extract_function_name_and_params(code):
    match = re.match(r'def (\w+)\s*\((.*?)\)', code)
    if match:
        function_name = match.group(1)
        params = match.group(2).split(',')
        params = [param.strip() for param in params]  # إزالة الفراغات الزائدة
        return function_name, params
    return None, None

df['function_name_and_params'] = df['cleaned_code'].apply(extract_function_name_and_params)

print(df[['cleaned_code', 'function_name_and_params']])


                                        cleaned_code function_name_and_params
0  def export_probabilityoutputname, kwargs\n    ...             (None, None)
1  def shared_symbol_tablename, version, symbols,...             (None, None)
2  def _read_para_hip_transport_modeself, code, c...             (None, None)
3  def setCheckable self, state \n        \n     ...             (None, None)
4  def from_moleculecls, molecule, labelsNone\n  ...             (None, None)


In [9]:
import re

def remove_comments(code):
    code = re.sub(r'#.*', '', code)
    return code

df['cleaned_code'] = df['cleaned_code'].apply(remove_comments)
print(df['cleaned_code'].to_list())

['def export_probabilityoutputname, kwargs\n    ...', 'def shared_symbol_tablename, version, symbols,...', 'def _read_para_hip_transport_modeself, code, c...', 'def setCheckable self, state \n        \n     ...', 'def from_moleculecls, molecule, labelsNone\n  ...']


In [10]:
def remove_empty_lines(code):
    code = '\n'.join([line for line in code.split('\n') if line.strip() != ''])
    return code

df['cleaned_code'] = df['cleaned_code'].apply(remove_empty_lines)
print(df['cleaned_code'].to_list())

['def export_probabilityoutputname, kwargs\n    ...', 'def shared_symbol_tablename, version, symbols,...', 'def _read_para_hip_transport_modeself, code, c...', 'def setCheckable self, state \n     ...', 'def from_moleculecls, molecule, labelsNone\n  ...']


In [11]:
pip install autopep8


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.4 MB/s eta 0:00:00


In [12]:
import autopep8

def format_code(code):
    return autopep8.fix_code(code)

df['cleaned_code'] = df['cleaned_code'].apply(format_code)
print(df['cleaned_code'].to_list())

['def export_probabilityoutputname, kwargs\n\n\n...\n', 'def shared_symbol_tablename, version, symbols, ...\n', 'def _read_para_hip_transport_modeself, code, c...\n', 'def setCheckable self, state\n\n\n...\n', 'def from_moleculecls, molecule, labelsNone\n\n\n...\n']


In [13]:
def remove_keywords(code):
    code = re.sub(r'\bself\b', '', code)
    return code

df['cleaned_code'] = df['cleaned_code'].apply(remove_keywords)


In [14]:
def clean_extra_whitespaces(code):
    code = re.sub(r'\s+', ' ', code)
    return code.strip()

df['cleaned_code'] = df['cleaned_code'].apply(clean_extra_whitespaces)
print(df['cleaned_code'].to_list())

['def export_probabilityoutputname, kwargs ...', 'def shared_symbol_tablename, version, symbols, ...', 'def _read_para_hip_transport_modeself, code, c...', 'def setCheckable , state ...', 'def from_moleculecls, molecule, labelsNone ...']


In [15]:
def convert_to_lowercase(code):
    return code.lower()

df['cleaned_code'] = df['cleaned_code'].apply(convert_to_lowercase)


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_code'])

X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

print(X_df.head())


   _read_para_hip_transport_modeself      code       def  \
0                           0.000000  0.000000  0.319302   
1                           0.000000  0.000000  0.265256   
2                           0.670092  0.670092  0.319302   
3                           0.000000  0.000000  0.319302   
4                           0.000000  0.000000  0.265256   

   export_probabilityoutputname  from_moleculecls    kwargs  labelsnone  \
0                      0.670092          0.000000  0.670092    0.000000   
1                      0.000000          0.000000  0.000000    0.000000   
2                      0.000000          0.000000  0.000000    0.000000   
3                      0.000000          0.000000  0.000000    0.000000   
4                      0.000000          0.556669  0.000000    0.556669   

   molecule  setcheckable  shared_symbol_tablename     state   symbols  \
0  0.000000      0.000000                 0.000000  0.000000  0.000000   
1  0.000000      0.000000               

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(df['cleaned_code'], test_size=0.2, random_state=42)

print(X_train.head(), X_test.head())


4       def from_moleculecls, molecule, labelsnone ...
2    def _read_para_hip_transport_modeself, code, c...
0         def export_probabilityoutputname, kwargs ...
3                         def setcheckable , state ...
Name: cleaned_code, dtype: object 1    def shared_symbol_tablename, version, symbols,...
Name: cleaned_code, dtype: object


In [18]:
df.to_csv('cleaned_code_data.csv', index=False)
